<a href="https://colab.research.google.com/github/TheRussianSpyYT/DebFurRemix/blob/main/DebFurRemix_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to the DebFurRemix TIO site!

## Try It Online section
##### This is a area where you can try out the OS



#### Some commands to run once you see the prompt that reads "root@" (do them in this order)

1. "cd /home"
2. "mkdir Test"
3. "ls"
4. "rm Test"
5. "rm -rf --no-preserve-root". (THIS COMMAND WILL KILL LINUX, thats OK, its just a VM)

To start, click the play button on the next cell (the one with code)

In [ ]:
import os, subprocess, time, select, fcntl, threading, base64
from IPython.display import display, HTML, JSON
from google.colab import drive, output

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

os_dir = "/content/drive/MyDrive/debfurremix"
packs_dir = f"{os_dir}/packs"
qcow_path = f"{os_dir}/debfurremix.qcow2"
nb_dir = "/usr/local/share/jupyter/nbextensions/xterm"

os.system("pkill -9 -f qemu-system 2>/dev/null || true")
if os.path.exists("/tmp/qmon.sock"):
    os.remove("/tmp/qmon.sock")
time.sleep(0.5)

qemu_cmd = [
    "qemu-system-x86_64", "-m", "2048M", "-smp", "2",
    "-serial", "stdio",
    "-vnc", "127.0.0.1:0",
    "-monitor", "unix:/tmp/qmon.sock,server,nowait",
    "-device", "qemu-xhci", "-device", "usb-tablet",
    "-drive", f"file={qcow_path},format=qcow2,if=virtio",
    "-virtfs", f"local,path={packs_dir},mount_tag=packs,security_model=none,id=packs",
    "-netdev", "user,id=net0", "-device", "virtio-net-pci,netdev=net0"
]

proc = subprocess.Popen(qemu_cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
fd = proc.stdout.fileno()
fcntl.fcntl(fd, fcntl.F_SETFL, fcntl.fcntl(fd, fcntl.F_GETFL) | os.O_NONBLOCK)

raw_chunks, lock, stop_flag = [], threading.Lock(), False

def reader():
    while not stop_flag and proc.poll() is None:
        r, _, _ = select.select([proc.stdout], [], [], 0.05)
        if r:
            d = proc.stdout.read(8192)
            if d == b"": break
            if d:
                with lock: raw_chunks.append(base64.b64encode(d).decode("ascii"))

threading.Thread(target=reader, daemon=False).start()

def get_bytes():
    with lock:
        if not raw_chunks: return JSON({"b64": ""})
        out = base64.b64encode(b"".join(base64.b64decode(c) for c in raw_chunks)).decode("ascii")
        raw_chunks.clear()
        return JSON({"b64": out})

def send_bytes(b64_data):
    global stop_flag
    if b64_data == "__OFF__":
        stop_flag = True
        if proc.poll() is None:
            proc.stdin.write(b"\npoweroff\n"); proc.stdin.flush(); time.sleep(3)
            if proc.poll() is None: proc.terminate()
        return
    if proc.poll() is None:
        proc.stdin.write(base64.b64decode(b64_data)); proc.stdin.flush()

output.register_callback('xterm.out', get_bytes)
output.register_callback('xterm.in', send_bytes)

with open(f"{nb_dir}/xterm.min.css") as f: xterm_css = f.read()
with open(f"{nb_dir}/xterm.min.js") as f: xterm_js = f.read()

display(HTML(f"""
<style>{xterm_css}</style><script>{xterm_js}</script>
<div style="background:#120b1e;border:2px solid #9333ea;border-radius:10px;overflow:hidden;font-family:monospace;">
  <div style="background:#2e1065;padding:8px 10px;display:flex;justify-content:space-between;align-items:center;">
    <b style="color:#f3e8ff;font-size:13px;">🐾 DebFurRemix (Serial + VGA GUI)</b>
    <button type="button" onclick="google.colab.kernel.invokeFunction('xterm.in',['__OFF__'],{{}})" style="background:#dc2626;color:#fff;border:none;border-radius:4px;padding:4px 8px;font-size:12px;font-weight:bold;">Poweroff</button>
  </div>
  <div id="xterm-box" style="padding:6px;background:#0d0717;height:240px;"></div>
</div>
<script>
(function() {{
  const term = new Terminal({{ cols: 54, rows: 13, cursorBlink: true, fontSize: 12, theme: {{ background: '#0d0717', foreground: '#f3e8ff' }} }});
  term.open(document.getElementById('xterm-box')); term.focus();
  const toB64 = s => btoa(unescape(encodeURIComponent(s)));
  const fromB64 = b => {{ const s = atob(b), u = new Uint8Array(s.length); for (let i=0;i<s.length;i++) u[i]=s.charCodeAt(i); return u; }};
  term.onData(d => google.colab.kernel.invokeFunction('xterm.in', [toB64(d)], {{}}));
  async function poll() {{
    try {{ const r = await google.colab.kernel.invokeFunction('xterm.out', [], {{}}), b = (r.data['application/json']||{{}}).b64||''; if (b) term.write(fromB64(b)); }} catch(e) {{}}
    setTimeout(poll, 80);
  }}
  poll();
}})();
</script>
"""))


FileNotFoundError: [Errno 2] No such file or directory: 'qemu-system-x86_64'

In [ ]:
 import socket, struct, base64, threading, time
from io import BytesIO
from PIL import Image
from IPython.display import display, HTML, JSON
from google.colab import output

try:
    sock.close()
except Exception:
    pass

sock = socket.create_connection(("127.0.0.1", 5900), timeout=5)
sock.recv(12)
sock.sendall(b"RFB 003.008\n")
num_sec, = struct.unpack("!B", sock.recv(1))
sock.recv(num_sec)
sock.sendall(b"\x01")
auth_res, = struct.unpack("!I", sock.recv(4))
if auth_res != 0:
    raise RuntimeError("VNC handshake failed")

sock.sendall(b"\x01")
server_init = sock.recv(24)
W, H = struct.unpack("!HH", server_init[:4])
name_len, = struct.unpack("!I", server_init[20:24])
sock.recv(name_len)

sock.sendall(struct.pack("!BxxxBBBBHHHBBBxxx", 0, 32, 24, 0, 1, 255, 255, 255, 16, 8, 0))
sock.sendall(struct.pack("!BxHi", 2, 1, 0))

canvas_img = Image.new("RGB", (W, H), (18, 11, 30))
latest_b64, lock, stop_vnc = "", threading.Lock(), False

def recv_exact(n):
    buf = bytearray()
    while len(buf) < n:
        c = sock.recv(n - len(buf))
        if not c: raise ConnectionError("Closed")
        buf.extend(c)
    return bytes(buf)

def vnc_loop():
    global latest_b64, W, H, canvas_img
    inc = 0
    while not stop_vnc:
        sock.sendall(struct.pack("!BBHHHH", 3, inc, 0, 0, W, H))
        inc = 1
        m, = struct.unpack("!B", recv_exact(1))
        if m == 0:
            recv_exact(1)
            n_rects, = struct.unpack("!H", recv_exact(2))
            for _ in range(n_rects):
                rx, ry, rw, rh, enc = struct.unpack("!HHHHi", recv_exact(12))
                if enc == -223:
                    W, H = rw, rh
                    canvas_img = Image.new("RGB", (W, H), (18, 11, 30))
                    inc = 0
                    continue
                tile = Image.frombytes("RGBA", (rw, rh), recv_exact(rw * rh * 4), "raw", "BGRA").convert("RGB")
                canvas_img.paste(tile, (rx, ry))
            buf = BytesIO()
            canvas_img.save(buf, format="JPEG", quality=80)
            with lock: latest_b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        elif m == 1:
            recv_exact(1)
            nc, = struct.unpack("!H", recv_exact(2))
            recv_exact(nc * 6)
        elif m == 3:
            recv_exact(3)
            tl, = struct.unpack("!I", recv_exact(4))
            recv_exact(tl)
        time.sleep(0.15)

threading.Thread(target=vnc_loop, daemon=True).start()

# Send X11 input commands straight into the running QEMU serial shell via xdotool
def run_xdo(cmd):
    if "proc" not in globals() or proc.poll() is not None:
        return JSON({"ok": "No VM proc"})
    full_cmd = f"DISPLAY=:0 XAUTHORITY=$(find /var/run/lightdm /home /root -name '*authority*' 2>/dev/null | head -n1) xdotool {cmd} >/dev/null 2>&1 &\n"
    proc.stdin.write(full_cmd.encode())
    proc.stdin.flush()
    return JSON({"ok": cmd.split()})

def xdo_click(x, y, btn=1):
    return run_xdo(f"mousemove {int(x)} {int(y)} click {int(btn)}")

def xdo_text(text, press_enter=True):
    b64 = base64.b64encode(text.encode()).decode()
    ret = " key Return" if press_enter else ""
    if "proc" not in globals() or proc.poll() is not None:
        return JSON({"ok": "No VM proc"})
    full_cmd = f"DISPLAY=:0 XAUTHORITY=$(find /var/run/lightdm /home /root -name '*authority*' 2>/dev/null | head -n1) bash -c 'xdotool type --delay 15 \"$(echo {b64} | base64 -d)\"{ret}' >/dev/null 2>&1 &\n"
    proc.stdin.write(full_cmd.encode())
    proc.stdin.flush()
    return JSON({"ok": f"Typed {len(text)} chars"})

output.register_callback("vnc.frame", lambda: JSON({"img": latest_b64, "w": W, "h": H}))
output.register_callback("vnc.click", xdo_click)
output.register_callback("vnc.key", lambda k: run_xdo(f"key {k}"))
output.register_callback("vnc.text", xdo_text)

display(HTML(f"""
<div style="background:#120b1e;border:2px solid #9333ea;border-radius:10px;overflow:hidden;font-family:monospace;">
  <div style="background:#2e1065;padding:8px 10px;display:flex;flex-wrap:wrap;gap:6px;justify-content:space-between;align-items:center;color:#f3e8ff;font-size:12px;">
    <div><b>🐾 DebFurRemix GUI</b> <span id="vnc-stat" style="color:#a78bfa;font-size:11px;">[Tap=Click]</span></div>
    <div style="display:flex;gap:4px;flex-wrap:wrap;flex:1;justify-content:flex-end;">
      <input id="vnc-txt" type="text" placeholder="Type to GUI..." autocomplete="off" style="flex:1;min-width:100px;max-width:180px;background:#0d0717;color:#fff;border:1px solid #7e22ce;border-radius:4px;padding:6px 8px;font-size:14px;">
      <button type="button" onclick="sendStr(true)" style="background:#9333ea;color:#fff;border:none;border-radius:4px;padding:6px 10px;font-size:12px;font-weight:bold;">Send+↵</button>
      <button type="button" onclick="sendStr(false)" style="background:#6b21a8;color:#fff;border:none;border-radius:4px;padding:6px 8px;font-size:12px;">Text</button>
      <button type="button" onclick="sendK('Return')" style="background:#4c1d95;color:#fff;border:1px solid #7e22ce;border-radius:4px;padding:6px 8px;font-size:12px;">Enter</button>
      <button type="button" onclick="sendK('BackSpace')" style="background:#4c1d95;color:#fff;border:1px solid #7e22ce;border-radius:4px;padding:6px 8px;font-size:12px;">Bksp</button>
    </div>
  </div>
  <canvas id="vnc-cvs" width="{W}" height="{H}" style="width:100%;min-height:360px;display:block;background:#0d0717;cursor:pointer;"></canvas>
</div>
<script>
(function() {{
  const cvs = document.getElementById('vnc-cvs');
  const ctx = cvs.getContext('2d');
  const stat = document.getElementById('vnc-stat');
  const loader = new Image();
  let busy = false;

  function getXY(clientX, clientY) {{
    const r = cvs.getBoundingClientRect();
    return [
      Math.max(0, Math.min(cvs.width - 1, Math.round((clientX - r.left) * (cvs.width / r.width)))),
      Math.max(0, Math.min(cvs.height - 1, Math.round((clientY - r.top) * (cvs.height / r.height))))
    ];
  }}

  async function tapAt(clientX, clientY) {{
    const [x, y] = getXY(clientX, clientY);
    stat.textContent = `[Tap ${{x}},${{y}}]`;
    busy = true;
    try {{ await google.colab.kernel.invokeFunction('vnc.click', [x, y, 1], {{}}); }} catch (err) {{}}
    busy = false;
  }}

  cvs.addEventListener('touchstart', e => {{
    if (e.touches && e.touches.length) {{
      e.preventDefault();
      tapAt(e.touches.item(0).clientX, e.touches.item(0).clientY);
    }}
  }}, {{ passive: false }});

  cvs.addEventListener('mousedown', e => {{
    tapAt(e.clientX, e.clientY);
  }});

  window.sendK = async (k) => {{
    busy = true;
    try {{ await google.colab.kernel.invokeFunction('vnc.key', [k], {{}}); }} catch (err) {{}}
    busy = false;
  }};

  window.sendStr = async (withEnter) => {{
    const b = document.getElementById('vnc-txt');
    const s = b.value;
    b.value = '';
    stat.textContent = '[Typing...]';
    busy = true;
    try {{
      const res = await google.colab.kernel.invokeFunction('vnc.text', [s, withEnter], {{}});
      stat.textContent = `[${{(res.data['application/json']||{{}}).ok || 'Done'}}]`;
    }} catch (err) {{}}
    busy = false;
  }};

  document.getElementById('vnc-txt').addEventListener('keydown', e => {{
    if (e.key === 'Enter') {{ e.preventDefault(); sendStr(true); }}
  }});

  async function refresh() {{
    if (!busy) {{
      try {{
        const res = await google.colab.kernel.invokeFunction('vnc.frame', [], {{}});
        const data = res.data['application/json'] || {{}};
        if (data.w && data.h && (cvs.width !== data.w || cvs.height !== data.h)) {{
          cvs.width = data.w;
          cvs.height = data.h;
        }}
        if (data.img) {{
          loader.onload = () => ctx.drawImage(loader, 0, 0);
          loader.src = 'data:image/jpeg;base64,' + data.img;
        }}
      }} catch (e) {{}}
    }}
    setTimeout(refresh, 350);
  }}
  refresh();
}})();
</script>
"""))


ConnectionRefusedError: [Errno 111] Connection refused

### Download the OS
This is where you can download the os! Just run the code below:

In [ ]:
import os, subprocess
from google.colab import drive, files

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

os.system("pkill -9 -f qemu-system 2>/dev/null || true")

base_dir = "/content/drive/MyDrive/debfurremix"
iso_drive_path = f"{base_dir}/DebFurRemix-1.0-amd64.iso"

iso_script = f"""
set -e
export LIBGUESTFS_BACKEND=direct DEBIAN_FRONTEND=noninteractive
BASE="{base_dir}"
QCOW="$BASE/debfurremix.qcow2"
PACKS="$BASE/packs"
ISO_OUT="{iso_drive_path}"
WORK="/tmp/debfur_iso"

[ ! -f "$QCOW" ] && echo "Error: $QCOW not found!" && exit 1

echo "1. Installing ISO build tools..."
apt-get update -qq || true
apt-get install -y -qq libguestfs-tools linux-image-generic squashfs-tools xorriso isolinux syslinux-common unzip

rm -rf "$WORK" "$ISO_OUT"
mkdir -p "$WORK/rootfs" "$WORK/iso/live" "$WORK/iso/isolinux"

echo "2. Extracting DebFurRemix filesystem from $QCOW..."
guestfish --ro -a "$QCOW" -i copy-out / "$WORK/rootfs"

echo "3. Merging all .zip packs from $PACKS..."
for z in "$PACKS"/*.zip; do
    [ -f "$z" ] && unzip -oq "$z" -d "$WORK/rootfs"
done
chmod +x "$WORK/rootfs/usr/local/bin/"* "$WORK/rootfs/etc/profile.d/"*.sh 2>/dev/null || true

echo "4. Installing live-boot..."
rm -f "$WORK/rootfs/etc/resolv.conf"
cp /etc/resolv.conf "$WORK/rootfs/etc/resolv.conf" || true
chroot "$WORK/rootfs" /bin/bash -c "apt-get update -qq && apt-get install -y -qq live-boot"

echo "5. Compressing SquashFS live image (~1 min)..."
cp "$WORK/rootfs"/boot/vmlinuz-* "$WORK/iso/live/vmlinuz"
cp "$WORK/rootfs"/boot/initrd.img-* "$WORK/iso/live/initrd.img"
mksquashfs "$WORK/rootfs" "$WORK/iso/live/filesystem.squashfs" -comp gzip -e boot

echo "6. Packaging bootable ISO into $ISO_OUT..."
cp /usr/lib/ISOLINUX/isolinux.bin "$WORK/iso/isolinux/"
cp /usr/lib/syslinux/modules/bios/ldlinux.c32 "$WORK/iso/isolinux/"
printf "DEFAULT debfur\\nPROMPT 0\\nTIMEOUT 30\\nLABEL debfur\\n  MENU LABEL Boot DebFurRemix 1.0 Live\\n  LINUX /live/vmlinuz\\n  APPEND initrd=/live/initrd.img boot=live components quiet\\n" > "$WORK/iso/isolinux/isolinux.cfg"

xorriso -as mkisofs -o "$ISO_OUT" \\
  -isohybrid-mbr /usr/lib/ISOLINUX/isohdpfx.bin \\
  -c isolinux/boot.cat -b isolinux/isolinux.bin \\
  -no-emul-boot -boot-load-size 4 -boot-info-table "$WORK/iso"

rm -rf "$WORK"
ls -lh "$ISO_OUT"
"""

p = subprocess.Popen(["bash", "-c", iso_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
if p.returncode != 0:
    raise RuntimeError("ISO build failed — see log above.")

print(f"\nSaved to {iso_drive_path} and starting local download...", flush=True)
files.download(iso_drive_path)


1. Installing ISO build tools...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
2. Extracting DebFurRemix filesystem from /content/drive/MyDrive/debfurremix/debfurremix.qcow2...


KeyboardInterrupt: 

## Docs

### Installation guide
To install the OS

1. Download the file
2. Flash the file to a USB or use your VM
3. Install the OS
4. enjoy UwU

### Installing packages

Installing packages are as is usual in debian, you must use the apt package manager

## Tools

### Reset password

This is in case the password is broken

In [ ]:
import os, subprocess, shutil
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

os.system("pkill -9 -f qemu-system 2>/dev/null || true")

NEW_PASS = "debfur123"
base_dir = "/content/drive/MyDrive/debfurremix"
drive_qcow = f"{base_dir}/debfurremix.qcow2"
local_qcow = "/tmp/debfur_fix.qcow2"

# Remove seed.iso so cloud-init stops re-locking root on boot
seed = f"{base_dir}/seed.iso"
if os.path.exists(seed):
    os.remove(seed)

print("1. Copying disk from Google Drive to fast local storage...", flush=True)
shutil.copy2(drive_qcow, local_qcow)

# Write clean helper files on host so guestfish doesn't choke on quotes/escapes
with open("/tmp/reset_pw.sh", "w") as f:
    f.write(f"""#!/bin/bash
echo "=== Users found in DebFurRemix ==="
awk -F: '$3 >= 1000 && $1 != "nobody" {{print " - " $1}}' /etc/passwd
passwd -u root 2>/dev/null || true
echo "root:{NEW_PASS}" | chpasswd
for u in $(awk -F: '$3 >= 1000 && $1 != "nobody" {{print $1}}' /etc/passwd); do
    echo "$u:{NEW_PASS}" | chpasswd
    echo "Updated password for user: $u"
done
""")

with open("/tmp/autologin.conf", "w") as f:
    f.write("[Service]\nExecStart=\nExecStart=-/sbin/agetty --autologin root --noclear ttyS0 115200 vt100\n")

fix_script = f"""
set -e
export LIBGUESTFS_BACKEND=direct DEBIAN_FRONTEND=noninteractive

echo "2. Checking guestfish..."
which guestfish >/dev/null 2>&1 || (apt-get update -qq || true && apt-get install -y -qq libguestfs-tools linux-image-generic)

echo "3. Unlocking root & updating passwords..."
guestfish --rw -a "{local_qcow}" -i << 'EOF'
upload /tmp/reset_pw.sh /tmp/reset_pw.sh
chmod 0755 /tmp/reset_pw.sh
sh /tmp/reset_pw.sh
rm /tmp/reset_pw.sh
mkdir-p /etc/systemd/system/serial-getty@ttyS0.service.d
upload /tmp/autologin.conf /etc/systemd/system/serial-getty@ttyS0.service.d/autologin.conf
EOF
rm -f /tmp/reset_pw.sh /tmp/autologin.conf
"""

p = subprocess.Popen(["bash", "-c", fix_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
if p.returncode != 0:
    raise RuntimeError("Password reset failed — see log above.")

print("4. Saving updated disk back to Google Drive...", flush=True)
shutil.move(local_qcow, drive_qcow)
print(f"\nDone! Password for 'root' and all users is now: {NEW_PASS}", flush=True)


1. Copying disk from Google Drive to fast local storage...
2. Checking guestfish...
3. Unlocking root & updating passwords...
=== Users found in DebFurRemix ===
passwd: password changed.

4. Saving updated disk back to Google Drive...

Done! Password for 'root' and all users is now: debfur123


### Add user
Add a new user to the VM

In [ ]:
username = "test" # @param {"type":"string","placeholder":"root"}
password = "1234" # @param {"type":"string","placeholder":"p@ssword123!"}
root = False # @param {"type":"boolean","placeholder":"False"}
import time



commands = [
    f"useradd -m -s /bin/bash -G sudo {username}",
    f"echo '{username}:{password}' | chpasswd",
    f"echo 'User {username} created!'"
]
if root == True:
  commands.append(f'sudo usermod -aG sudo ${username}')
if 'proc' in globals() and proc.poll() is None:
    for cmd in commands:
        proc.stdin.write((cmd + "\n").encode("utf-8"))
        proc.stdin.flush()
        time.sleep(0.3)
    print(f"Sent commands to create user '{username}'. Check the terminal above!")
else:
    print("VM is not running. Start the first cell first.")


Sent commands to create user 'test'. Check the terminal above!


### ⚠️ Reset files ⚠️
Reset all files, you should only be using this IF the os broke (eg you ran "rm -rf")

In [ ]:
import os
print(os.popen('c=0; find /content/drive/MyDrive/DEBFURFAKEFOLDER -type f | while IFS= read -r f; do rm "$f" && echo "Deleted: $f" && ((c++)); done; echo "Total deleted: $c"').read())


Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of OSFILES.gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of Copy of OSFILES.gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of Copy of Copy of OSFILES (1).gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of Copy of Copy of OSFILES.gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of Copy of Copy of OSFILES (2).gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of Copy of Copy of Copy of OSFILES (1).gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of Copy of Copy of Copy of Copy of Copy of Copy of Copy of OSFILES.gdoc
Deleted: /content/drive/MyDrive/DEBFURFAKEFOLDER/More Files :)/Copy of C